In [1]:
from platform import python_version
print(python_version())

3.11.14


### Calculating all possible Enrichment Analysis
  - for each LFC/FDR cutoff one calculates the Enrichment Analysis
  - We used Enricher python API
     - Reactome (2024)  <<--------------- only Reactome
     - Bioplanet (2019)
     - WikiPathways (2021 Human)
     - KEGG (2021 Human)
     - GO Biological Process (2021)
     - MSigDB Hallmark (2020)
   
### For each enriched pathways one calculates:
  - DEGs in the pathway
  - DEGs not in the pathway
  - TOI1, 2, 3, 4

In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"
ROOT_COLAB = ROOT0 / "colab"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import pdwritecsv, pdreadcsv, create_dir
from libs.enricher_lib import enricheR
from libs.dashcyto_lib import DASH_CYTO
from libs.calc_degs_lib import CALC_DEGS
from libs.config_lib import Config

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages
  warnings.warn(


In [3]:
root0 = Path(dic_yml['root0'])
root0_data = Path(dic_yml['root0_data'])
root_colab = root0 / 'colab'

email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"


root_project = root0_data / s_project
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'

disease = PSI_ID

root_project = create_dir(root0_data, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']

case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=root0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [6]:
enr = enricheR(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, root0=root0, root0_data=root0_data,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", enr.root0, enr.root_disease)
case = case_list[0]
print(">>>", case)

enr.cfg.set_default_best_lfc_cutoff(enr.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, prompt_verbose=False, verbose=False)
# print("\nEcho Parameters:")
# print(enr.echo_parameters())


Start opening tables ....
Building synonym dictionary ...

Start opening tables ....
Building synonym dictionary ...

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/TCGA-CESC
>>> Tumor


In [7]:
enr.LFC_cut_inf, LFC_cut_inf

(0.4, 0.4)

In [8]:
print(len(enr.gene.df_my_gene))
enr.gene.df_my_gene.head(2)

27223


,entrezid,symbol,geneid,name,synonyms,other_names,ec_enzyme,refseq_gen,refseq_prot,refseq_rna,...,acessions_rna,acessions_trans,map_location,dic_panther,ortholog,dic_uniprot,swissprot,wikipedia,refseq_gen,refseq_rna
0,ENSG00000121410,A1BG,1,alpha-1-B glycoprotein,"['A1B', 'ABG', 'GAB', 'HYST2477']","['HEL-S-163pA', 'alpha-1B-glycoprotein', 'epididymis secretory sperm binding...",NaN,NaN,NP_570602.2,NaN,...,"['AA484435.1', 'AB073611.1', 'AF414429.1', 'AI022193.1', 'AK055885.1', 'AK05...","[{'protein': 'AAH35719.1', 'rna': 'BC035719.1'}, {'protein': 'BAG51591.1', '...",19q13.43,"{'HGNC': '5', '_license': 'http://pantherdb.org/tou.jsp', 'ortholog': [{'MGI...","[{'MGI': '2152878', 'ortholog_type': 'LDO', 'panther_family': 'PTHR11738', '...","{'Swiss-Prot': 'P04217', 'TrEMBL': ['M0R009', 'V9HWD8']}",P04217,{'url_stub': 'A1BG (gene)'},"['NC_000019.10', 'NC_060943.1']",NM_130786.4
1,ENSG00000268895,A1BG-AS1,503538,A1BG antisense RNA 1,"['A1BG-AS', 'A1BGAS', 'NCRNA00181']","['A1BG antisense RNA (non-protein coding)', 'A1BG antisense RNA 1 (non-prote...",NaN,NaN,NaN,NaN,...,"['AK027222.1', 'BC006144.1', 'BC040926.1', 'NR_015380.2']",NaN,19q13.43,NaN,NaN,NaN,NaN,NaN,"['NC_000019.10', 'NC_060943.1']",NR_015380.2


In [9]:
lista = [x for x in os.listdir(enr.root_result) if 'medulloblastoma_DEG_' in x and not '~lock' in x]
lista.sort()
print(len(lista))
lista[:3]

0


[]

In [10]:
files = [x for x in os.listdir(enr.root_enrich) if 'Reactome_' in x and not '~lock' in x and '_WNT_' in x]
print(len(files))
files[:2]

0


[]

### Summary of cases - below on can see the enriched tables for different databases

In [11]:
case_list

['Tumor']

In [13]:
print("")

for case in case_list:
    enr.open_case(case, verbose=False)
    print(enr.echo_parameters())
    print("\n------------------\n\n")


	20006/20006 DEGs/ensembl.
		Up 10358/10358 DEGs/ensembl.
		Dw 9648/9648 DEGs/ensembl.

Found 0 (best=3) pathways for geneset num=0 'Reactome_Pathways_2024'
Pathway cutoffs p-value=0.050 fdr=0.050 min genes=0.05No enrichment analysis was calculated.

------------------




### Cutoffs and Results

In [15]:
for case in case_list:
    ret, degs, degs_ensembl, dfdegs = enr.open_case(case, verbose=False)

    print(f"For {case}")
    print(f"\tLFC cutoffs: lfc={enr.LFC_cut:.3f}; fdr={enr.lfc_FDR_cut} #{enr.s_deg_dap}s = {len(degs)}")
    # print(f"\tPathway cutoffs: fdr={enr.fdr_pathway_cutoff:.3f}; num of genes={enr.num_of_genes_cutoff}, #Pathways = {enr.n_pathways}, #{enr.s_deg_dap}s in pathwyas = {enr.n_degs_in_pathways}\n")


For Tumor
	LFC cutoffs: lfc=0.900; fdr=1 #DEGs = 20006


In [16]:
print(">>> lfc_FDR_cut", enr.lfc_FDR_cut)

try:
    dflfc = enr.dflfc_ori[(enr.dflfc_ori.fdr < enr.lfc_FDR_cut)]
    print(len(dflfc))
except:
    dflfc = pd.DataFrame()

dflfc.head(3)

>>> lfc_FDR_cut 1
31198


,ensembl_id,symbol,biotype,lfc,lfcSE,statistic,pval,fdr,baseMean,method,abs_lfc
1,ENSG00000179046,TRIML2,protein_coding,21.789,3.100,7.028,2.090e-12,4.906e-11,28.213,DESeq2,21.789
2,ENSG00000205325,AC005863.1,lncRNA,20.968,3.741,5.606,2.076e-08,2.626e-07,16.533,DESeq2,20.968
3,ENSG00000269968,AC006064.4,lncRNA,19.504,1.339,14.561,4.989e-48,1.661e-45,57092.293,DESeq2,19.504


In [ ]:
# df2 = enr.dflfc_ori[ (enr.dflfc_ori.symbol == 'IGHA2') | (enr.dflfc_ori.symbol == 'A2M')]
# df2

In [17]:
fname_final_ori, fname_ori, title = enr.set_lfc_names()
fname_final_ori, title

('TCGA-CESC_final_LFC_Tumor_x_CTRL_not_normalized.tsv',
 'Tumor (not_normalized)')

In [ ]:
enr.set_enrichment_name()

In [ ]:
enr.get_best_ptw_cutoff_biopax(verbose=True)
# self.pathway_pval_cutoff, self.fdr_pathway_cutoff, self.num_of_genes_cutoff,

### Testing EnrichR API 

In [ ]:
case = case_list[0]
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, verbose=False)
ret, len(degs), enr.n_degs, enr.n_degs_ensembl

In [ ]:
# dfdegs.columns

In [ ]:
print(len(dfdegs))
dfdegs.head(3)

### Biotypes

https://www.ensembl.org/info/genome/genebuild/biotypes.html

  - TEC (To be Experimentally Confirmed)


In [ ]:
dfdegs_ensembl = dfdegs[ (~pd.isnull(dfdegs.ensembl_id)) & (dfdegs.biotype != 'TEC')].copy()
cols = ['probe', 'symbol', 'symbol_prev', 'symb_or_syn', 'biotype', '_type',
       'lfc', 'abs_lfc', 'fdr', 'description',
       'desc_gff', 'description_prev', 'accession', 'ensembl_id',
       'ensembl_transc_id', 'geneid', 'cytoband', 'symbol_pipe', ]
print(len(dfdegs), len(enr.dflfc), len(dfdegs_ensembl), len(enr.dfdegs_ensembl))
dfdegs_ensembl[cols].head()

In [ ]:
np.unique(dfdegs_ensembl.biotype)

In [ ]:
len(enr.degs), len(enr.degs_ensembl)

In [ ]:
enr.n_degs, enr.n_degs_ensembl

In [ ]:
enr.set_db(geneset_num=0)

In [ ]:
shortId, userListId = enr.open_session_upload_symbols(enr.degs_ensembl)
shortId, userListId

### All enriched cases for many databases

In [ ]:
fdr_ptw_cutoff_list = enr.fdr_ptw_cutoff_list
fdr_ptw_cutoff_list

In [ ]:
# dfsim = pdreadcsv( enr.cfg.fname_lfc_cutoff, enr.cfg.root_config)
dfsim = enr.open_simulation_table()
if dfsim is None:
    dfsim = pd.DataFrame()

print(len(dfsim))
dfsim.tail(3)

In [ ]:
enr.lfc_list

In [ ]:
enr.fdr_list

### How many samples per case?

In [ ]:
for case in case_list:
    dfsim2 = dfsim[ (dfsim.case == case) & (dfsim.normalization == enr.normalization) & 
                    (dfsim.n_degs >= enr.num_min_degs_for_ptw_enr)]
    print(f"case {case} #simulations {len(dfsim2)}")

In [ ]:
dfsim2 = dfsim[ (dfsim.normalization == enr.normalization) & (dfsim.n_degs > 2)].copy()
dfsim2.index = np.arange(0, len(dfsim2))
print(len(dfsim2))

In [ ]:
for i in range(len(dfsim2)):
    row = dfsim2.iloc[i]
    degs = eval(row.degs)

    case = row.case
    LFC_cut = row.LFC_cut
    lfc_FDR_cut = row.lfc_FDR_cut

    print(i, case, LFC_cut, lfc_FDR_cut, len(degs), degs[:9], '...')
    if i > 3: break

In [ ]:
enr.LFC_cut_inf

In [ ]:
dfsim[ (dfsim.case == 'WNT') & (dfsim.LFC_cut == LFC_cut_inf) & (dfsim.lfc_FDR_cut == 0.15)]

In [ ]:
dfsim[ (dfsim.case == 'G4') & (dfsim.LFC_cut == LFC_cut_inf) & (dfsim.lfc_FDR_cut == 0.15)]

### Calc all enrichment analyses

In [ ]:
geneset_num_list = [1, 2, 4, 5, 7]
geneset_num_list = [0, 1, 2, 4, 5, 7]
geneset_num_list = [0]

In [ ]:
enr.set_db(0, verbose=True)

In [ ]:
enr.set_enrichment_name()

In [ ]:
enr.LFC_cut_inf

In [ ]:
print(enr.LFC_cut_inf)
df_fdr = enr.open_fdr_lfc_correlation(case, enr.LFC_cut_inf)

In [ ]:
dfsim = enr.open_simulation_table()
dfsim.head(3)

### Calc DEFAULT paramenters Enrichment Analysis

In [ ]:
force = False
verbose = False
enr.calc_default_enrichment_analysis(geneset_num_list=[0, 1, 2, 4, 5, 7], force=force, verbose=verbose)

### Reactome in Enricher run in 2024-03-10

In [ ]:
enr.LFC_cut_inf

In [ ]:
case = case_list[1]

LFC_cut_inf = 1
LFC_cut_inf = enr.LFC_cut_inf

df_fdr = enr.open_fdr_lfc_correlation(case, LFC_cut_inf)

try:
    df2 = df_fdr[ pd.notnull(df_fdr['corr']) ]
    print(len(df2))
except:
    df2 = pd.DataFrame()

print(len(df2))
df2

In [ ]:
df_fdr = enr.open_fdr_lfc_correlation(case, enr.LFC_cut_inf)
df_fdr.head(2)

In [ ]:
verbose = False
geneset_num_list = [0]
# remove the comments - it last some minutes
enr.calc_all_enrichment_analysis(geneset_num_list, iecho=50, force=force, verbose=verbose)

In [ ]:
verbose = False
geneset_num_list = [1, 2, 4, 5, 7]
# remove the comments to run - it lasts some minutes
enr.calc_all_enrichment_analysis(geneset_num_list, iecho=50, force=force, verbose=verbose)

### Sampling Pathways 

In [ ]:
dfa = enr.count_sampling(geneset_num_list=[0], prompt_verbose=True)
len(dfa)

In [ ]:
fig, dfa = enr.barplot_sampling_cutoffs(prompt_verbose=False, verbose=False)
fig.show()

### Differences between databases
#### Run only if you defined teh best config: new05 algorithm

In [ ]:
enr.get_best_ptw_cutoff_biopax()

In [ ]:
case = case_list[0]

In [ ]:
enr.LFC_cut , enr.lfc_FDR_cut, enr.pathway_pval_cutoff, enr.fdr_pathway_cutoff, enr.num_of_genes_cutoff

In [ ]:
case = case_list[0]
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, verbose=False)
len(degs)

In [ ]:
fname, fname_cutoff = enr.set_enrichment_name()
fname, fname_cutoff 

In [ ]:
geneset_num_list = [0, 1, 2, 4, 5, 7]

for geneset_num in geneset_num_list:
    enr.set_db(geneset_num, verbose=True)

### Reactome_2022

In [ ]:
enr.set_db(0, verbose=True)
case = case_list[0]
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, verbose=False)
# print("\nEcho Parameters:")
enr.echo_parameters()

In [ ]:
if enr.df_enr is None:
    enr.df_enr = pd.DataFrame()
print(len(enr.df_enr))
enr.df_enr

### Reactome_2022 case G4

In [ ]:
enr.set_db(0, verbose=True)
case = case_list[1]
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, verbose=False)
# print("\nEcho Parameters:")
enr.echo_parameters()

In [ ]:
if enr.df_enr is None:
    enr.df_enr = pd.DataFrame()
print(len(enr.df_enr))
enr.df_enr

### WikiPathway_2021_Human

In [ ]:
enr.set_db(1, verbose=True)
case = case_list[0]
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, verbose=False)

In [ ]:
enr.echo_enriched_pathways()

In [ ]:
if enr.df_enr is None:
    enr.df_enr = pd.DataFrame()

print(len(enr.df_enr))
enr.df_enr.head(42)

In [ ]:
enr.df_enr.tail(40)

### KEGG

In [ ]:
enr.set_db(2, verbose=True)
case = case_list[0]
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, verbose=False)
enr.echo_enriched_pathways()

In [ ]:
if enr.df_enr is None:
    enr.df_enr = pd.DataFrame()

print(len(enr.df_enr))
enr.df_enr.head(45)

In [ ]:
enr.df_enr.tail(40)

In [ ]:
enr.set_db(2, verbose=True)
case = case_list[1]
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, verbose=False)
enr.echo_enriched_pathways()

In [ ]:
if enr.df_enr is None:
    enr.df_enr = pd.DataFrame()

enr.df_enr.head(30)

In [ ]:
enr.df_enr.tail(30)

### BioPlanet_2019 = 4

In [ ]:
enr.set_db(4, verbose=True)
case = case_list[0]
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, verbose=False)
enr.echo_enriched_pathways()

In [ ]:
if enr.df_enr is None:
    enr.df_enr = pd.DataFrame()

print(len(enr.df_enr))
enr.df_enr.head(57)

In [ ]:
enr.df_enr.tail(50)

In [ ]:
enr.set_db(4, verbose=True)
case = case_list[1]
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, verbose=False)
enr.echo_enriched_pathways()

In [ ]:
if enr.df_enr is None:
    enr.df_enr = pd.DataFrame()

print(len(enr.df_enr))
enr.df_enr.head(50)

In [ ]:
enr.df_enr.tail(43)

### Development & tests

In [ ]:
force=True; verbose=False
num_min_degs_for_ptw_enr = 3

geneset_num_list = [1, 2, 4, 5, 7]
geneset_num_list = [0, 1, 2, 4, 5, 7]
geneset_num_list = [0]

icount=-1
for case in case_list:
    if not enr.open_case_simple(case):
        print(f"Problems for {case} !!!!")
        continue
    
    dfsim2 = dfsim[ (dfsim.normalization == enr.normalization) & (dfsim.case == case) &
                    (dfsim.n_degs >= num_min_degs_for_ptw_enr)]
    
    for i in range(len(dfsim2)):
        icount += 1
        
        row = dfsim2.iloc[i]

        degs = eval(row.degs)
        case = row.case
        
        LFC_cut = row.LFC_cut
        lfc_FDR_cut = row.lfc_FDR_cut

        degs2, _ = enr.list_of_degs_params(LFC_cut, lfc_FDR_cut, verbose=False)

        if len(degs) != len(degs2):
            print("Error:", case, LFC_cut, lfc_FDR_cut, len(degs), len(degs2))
            continue

        # if i > 10:break
        enr.calc_EA_dataset_symbol(degs, return_value=True, force=force, verbose=verbose)
        if icount%100==0:
            print(case, len(degs), LFC_cut, lfc_FDR_cut)
            enr.echo_degs()
            print("")
            enr.echo_enriched_pathways()
            print("\n")


### Reactome, Bioplanet, KEGG

In [ ]:
enr.dbs_list

In [ ]:
[enr.dbs_list[i] for i in [0, 1, 2, 4, 5, 7]]

In [ ]:
enr.set_which_db('xxx')

In [ ]:
enr.set_which_db('Reactome_2022')

In [ ]:
enr.set_which_db('Reactome')

In [ ]:
enr.set_which_db('reactome')

In [ ]:
enr.set_which_db('KEGG_2021')

In [ ]:
enr.set_which_db('KEGG')

In [ ]:
enr.set_db(geneset_num = 0)

### Start with Reactome

In [ ]:
case_list

In [ ]:
case = case_list[0]
enr.open_case(case, verbose=False)
degs, dfdegs = enr.list_of_degs(force=False, verbose=True)

# enr.set_which_db('Reactome_2022')
geneset_num = 0
force=False; verbose=False

print(f"case: {case}")
print(f"there are {len(degs)} for fdr < {enr.lfc_FDR_cut:.3f} and lfc >= {enr.LFC_cut:.3f}")
print(f"Pathway filter fdr < {enr.fdr_pathway_cutoff:.3f} and lfc >= {enr.pval_pathway_cutoff:.3f} and num_of_genes_cutoff={enr.num_of_genes_cutoff}")

enr.calc_EA_dataset_symbol(degs, force=force, verbose=verbose)

In [ ]:
i=0
case = case_list[i]
ret, degs, dfdegs = enr.open_case(case, verbose=False)

enr.df_enr

In [ ]:
print(enr.geneset_lib)

row = enr.get_enriched_pathway_line(i_line=0)

if row is not None:
    print(row.pathway, row.pathway_id)
    print(len(enr.genes_in_pathway), enr.genes_in_pathway)


### Enriched DEGs

In [ ]:
len(enr.all_enr_degs), ",".join(enr.all_enr_degs)

In [ ]:
",".join(enr.degs)

In [ ]:
enr_found_degs = [x for x in enr.degs if x in enr.all_enr_degs]
",".join(enr_found_degs)

### Found genes 

In [ ]:
len(enr.all_enr_degs), ",".join(enr.all_enr_degs)

In [ ]:
len(enr.enr_found_degs), ",".join(enr.enr_found_degs)

### Not found genes in pathways

In [ ]:
len(enr.enr_not_found_degs), ",".join(enr.enr_not_found_degs)

### BioPlanet

In [ ]:
enr.set_db(geneset_num=4)

In [ ]:
dfbiop = enr.open_db_pathway()
print(len(dfbiop))
dfbiop.head(3)

### Bioplanet diseases

In [ ]:
_ = enr.open_bioplanet_disease(force=False)
dfdisease = enr.dfdisease
print(len(dfdisease))
dfdisease.head()

### Bioplanet Categories

In [ ]:
 _ = enr.open_bioplanet_category(force=False)
dfbiop_cat = enr.dfbiop_cat
print(type(dfbiop_cat.category))
print(len(dfbiop_cat))
dfbiop_cat.head()

In [ ]:
def all_equal_list(cols1, cols2):
    if cols1 is None and cols2 is None: return True
    if cols1 == [] and cols2 == []: return True
    
    if len(cols1) != len(cols2): return False
    
    soma = np.sum([1 if cols1[i] != cols2[i] else 0 for i in range(len(cols1))])
    return soma == 0

cols1 = list(enr.dflfc_all.columns)

cols2 = ['probe', 'symbol', 'geneid', 'description', 'logFC', 'meanExpr', \
        't.stat', 'p-value', 'fdr', 'B', 'chr.range', 'org.chromosome', \
        'forward.reverse', 'nuc.sequence', 'gemmaid', 'go.term']

all_equal_list(cols1, cols2)

In [ ]:
all_genes = []
for i in range(len(dfsig)):
    genes = eval(dfsig.iloc[i].genes)
    # print(i, len(genes))
    all_genes += genes
    
all_genes = np.unique(all_genes)
all_genes.sort()
all_genes

not_found_genes = np.array([x for x in enr.deg_list if not x in all_genes])
not_found_genes

In [ ]:
def find_hugo_symbol(gene):
    try:
        i = enr.gene.dic_genes[gene]
        if isinstance(i, list):
            i = i[0]
            
        mat = enr.gene.df_synonyms.iloc[i]['synonyms']
        print(">>>", gene, i, mat, type(mat))
        if isinstance(mat, str):
            mat = eval(mat)
            
        gene0 = mat[0]
    except:
        gene0 = gene

    return gene0

In [ ]:
gene = 'SEPP1'
find_hugo_symbol(gene)

In [ ]:
enr.dic_genes[gene]

In [ ]:
lista = [x for x in os.listdir(enr.root_result) if 'taubate_PAC_UP_' in x and not '~lock' in x ]
print(len(lista))
", ".join(lista)


In [ ]:
root = enr.root_result
_type = '.tsv'
_type = None
pattern_src = 'taubate_PAC'
pattern_dst = 'taubate_MAP'

if _type is None:
    lista = [x for x in os.listdir(root) if pattern_src in x and not '~lock' in x ]
else:
    lista = [x for x in os.listdir(root) if pattern_src in x and not '~lock' in x and x.endswith(_type)]

if lista == []:
    print(f"There are no files in {root}")

for fname in lista:
    fname_src = os.path.join(root, fname)

    fname_dst = fname.replace(pattern_src, pattern_dst)
    fname_dst = os.path.join(root, fname_dst)

    if not  os.path.exists(fname_src):
        print(f"fname does not exist: '{fname_src}'", fname_src)
        continue
    if os.path.exists(fname_dst):
        print(f"fname already exists: '{fname_dst}'")
        continue
        
    if fname_dst == fname_src:
        print(f"fname did not change: 'fname_src'")
        continue
        
    print(fname_src, 'to', fname_dst)
    os.rename(fname_src, fname_dst)

<p style="font-size: 24px; color: cyan;">
$Hypergeometric Probability = p(k,M, n, N) = \frac{\binom{n}{k} * \binom{M-n}{N-k}}{\binom{M}{N}}$
</p>

https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.hypergeom.html

### The Super-Hypergeometric statistics
  - given all enriched pathways
  - given all DEGs = n
  - given all degs in pathways = k

  - given all genes declared in the experiment (microarray for example) - n_total_genes = M
  - given all genes annotated in the enriched pathways (n_annotated) = N

  - what is the possibility to find n_degs_in_pathays given that M, n, N




In [ ]:
from scipy.stats import hypergeom

In [ ]:
n = 900 # DEGs
M = 2000 # Microarray
N = 300  # annottated in pathways
k = 150  # degs in pathways

In [ ]:
p = hypergeom.cdf(k, M, n, N)
if p >= .9:
    p = 1 - p
else:
    p

p

### Testing

In [ ]:
case  = case_list[0]
enr.open_case(case, verbose=False)

df_enr = enr.merge_reactome(enr.df_enr)
df_enr.head()

<p style="font-size: 24px; color: cyan;">
$Hypergeometric Probability = p(k,M, n, N) = \frac{\binom{n}{k} * \binom{M-n}{N-k}}{\binom{M}{N}}$
</p>

https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.hypergeom.html

In [ ]:
n = enr.n_degs # DEGs
M = enr.n_total_genes # Microarray

if enr.calc_all_annoted_genes():
    N = enr.n_annotated_genes # annottated in pathways
else:
    N = 0
    
k = enr.n_degs_in_pathways  # degs in pathways

print(f"There are {N} genes annotated in pathways and {M} genes available in the microarray experiment")
print(f"We found {k} DEGs in the enriched pathways in {n} DEGs obtained by new algorithm and cutoffs")

p = hypergeom.cdf(k, M, n, N)
if p >= .9:
    p = 1 - p
else:
    p

if p < 0.05:
    print(f"We reject H0, is statistically unlikely to find {k} DEGs in the pathways, p-value = {p:.3e}")
else:
    print(f"We accept H0, is statistically likely to find {k} DEGs in the pathways, p-value = {p:.3e}")
    

In [ ]:
n = 3   # pathway having {n} word
M = 40000 # total words
N = 30 # pathways' words in teh bag
k = 1  # degs in pathways

print(f"Tere are {N} pathways' words in the bag of {M} words in MB - WNT")
print("")

for pathway, k, n in [('path1', 1, 2), ('path2', 2, 5), ('path3', 3, 8)]:
    print(f"For pathway '{pathway}', we found {k} word(s) in the pathway having {n} words")
    
    p = hypergeom.cdf(k, M, n, N)
    if p >= .9:
        p = 1 - p
    else:
        p
    
    if p < 0.05:
        print(f"We reject H0, is statistically unlikely to find {k} DEGs in the pathways, p-value = {p:.3e}")
    else:
        print(f"We accept H0, is statistically likely to find {k} DEGs in the pathways, p-value = {p:.3e}")

    print("")

### Testing curation
#### first WNT

In [ ]:
n = 2100 # papers in Medulloblastom
M = 48151 # Pubmed Papers related to Neuronal Cancer 
N = 38  # pathways related to WNT Medulloblastoma, the come from the Literature
k = 23  # pathways for WNT found by curations

case2 = 'WNT'
print(f"For {case2}")
print(f"There are {N} enriched pathways in {case2} Medulloblastoma and {M} Pubmed Papers related to Neuronal Cancer ")
print(f"We found {k} pathways for {case2} found by curations in {n} papers in Medulloblastom")

p = hypergeom.cdf(k, M, n, N)
if p >= .9:
    p = 1 - p
else:
    p

if p < 0.05:
    print(f"We reject H0, is statistically unlikely to curate {k} pathways, p-value = {p:.3e}")
else:
    print(f"We accept H0, is statistically likely to curate {k} pathways, p-value = {p:.3e}")


In [ ]:
k = 12  # pathays for G4 found by curations

case2 = 'G4'
print(f"For {case2}")
print(f"There are {N} enriched pathways in {case2} Medulloblastoma and {M} Pubmed Papers related to Neuronal Cancer ")
print(f"We found {k} pathways for {case2} found by curations in {n} papers in Medulloblastom")

p = hypergeom.cdf(k, M, n, N)
if p >= .9:
    p = 1 - p
else:
    p

if p < 0.05:
    print(f"We reject H0, is statistically unlikely to curate {k} pathways, p-value = {p:.3e}")
else:
    print(f"We accept H0, is statistically likely to curate {k} pathways, p-value = {p:.3e}")


In [ ]:
def binomial_find_nDEGs_in_pathways():
    from scipy.stats import binom

p = 0.5
n = len(df2)
x = df2['found'].sum()
x, n


prob = binom.cdf(x, n, p)

if prob <= 0.5:
    pval = prob
        
    if pval < 0.05:
        print(f"Alternative Hypothesis: this finding could shows that the algoritm did a BAD JOB: pval = {pval:.3f}")
    else:
        print(f"Null Hypothesis: this finding could be achieved by random: pval = {pval:.3f}")
else:
    pval = 1 - prob
    
    if pval < 0.05:
        print(f"Alternative Hypothesis: this finding could not be achieved by random: pval = {pval:.3f}")
    else:
        print(f"Null Hypothesis: this finding could be achieved by random: pval = {pval:.3f}")

print(f"The curation found {x} pathways/concepts in {n} possible given pathways")
